# SWOT-Bot model benchmark, step by step

One cell per model, run top to bottom. Each cell:

1. loads the model's saved per-fold results from `RESULTS_DIR/folds/` if they exist (nothing is retrained);
2. otherwise, when `RUN_MISSING = True`, trains and evaluates the model on the shared 5 folds, saves the scores, and shows them;
3. on an error, prints the traceback in that cell and carries on, so the problem can be fixed and that cell rerun alone.

Results are saved per model, so a kernel crash (out of memory, GPU fault) loses only the model that was running.
The same functions back `benchmarks/benchmark.py`.

In [1]:
import os, sys, time, traceback
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("SWOTBOT_BENCH_OUT", str(ROOT / "benchmarks"))

# Train models that have no saved results. Keep False on CPU: BERT-size models take hours there.
RUN_MISSING = os.environ.get("RUN_MISSING", "0") == "1"

import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from benchmarks import benchmark as B

X, y = B.load_dataset(ROOT / "data" / "train.csv")
X, y = X.reset_index(drop=True), y.reset_index(drop=True)
SPLITS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y))
REG = B.registry()
print(f"{len(X)} passages, {len(REG)} models, results in {B.FOLDS_DIR}, RUN_MISSING={RUN_MISSING}")
print(y.value_counts().to_string())

2500 passages, 28 models, results in /home/user/swot-bot/benchmarks/folds, RUN_MISSING=False
STRENGTHS        827
WEAKNESSES       619
OPPORTUNITIES    549
THREATS          505


In [2]:
def step(name):
    """Load or run one model and show its per-fold scores. Errors are printed, not raised."""
    suite, family, factory = REG[name]
    path = B.FOLDS_DIR / f"{B.slug(name)}.csv"
    if path.exists():
        df, how = pd.read_csv(path), "loaded saved results"
    elif not RUN_MISSING:
        print(f"{name}: no saved results (set RUN_MISSING=1 to train it)")
        return None
    else:
        t0 = time.perf_counter()
        try:
            df = B.run_model(name, factory, X, y, SPLITS, family)
        except Exception:
            print(f"{name}: FAILED")
            traceback.print_exc()
            return None
        how = f"trained in {time.perf_counter() - t0:.0f}s"
    print(f"{name}  [{family}]  {how}")
    print(f"macro-F1 {df.f1_macro.mean():.4f} ± {df.f1_macro.std():.4f}   accuracy {df.accuracy.mean():.4f}   "
          f"log loss {df.log_loss.mean():.3f}   fit {df.fit_s.mean():.1f} s/fold   predict {df.ms_per_doc.iloc[0]:.2f} ms/doc")
    display(df[["fold", "f1_macro", "accuracy", "log_loss", "fit_s"]].round(4))
    return df

## Linear (TF-IDF)

In [3]:
_ = step('TF-IDF + LogReg (current production)')

TF-IDF + LogReg (current production)  [Linear (TF-IDF)]  loaded saved results
macro-F1 0.8446 ± 0.0089   accuracy 0.8460   log loss 0.452   fit 5.7 s/fold   predict 0.10 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8313,0.834,0.4813,5.0997
1,1,0.8474,0.852,0.4441,7.0891
2,2,0.8457,0.848,0.4588,5.2100
3,3,0.8424,0.840,0.4226,4.8071
4,4,0.8559,0.856,0.4527,6.1503


In [4]:
_ = step('TF-IDF word+char + LogReg')

TF-IDF word+char + LogReg  [Linear (TF-IDF)]  loaded saved results
macro-F1 0.8477 ± 0.0155   accuracy 0.8492   log loss 0.422   fit 8.1 s/fold   predict 0.71 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8348,0.838,0.4534,7.3304
1,1,0.8433,0.848,0.4031,6.3687
2,2,0.8337,0.836,0.4378,9.3879
3,3,0.8570,0.854,0.3983,8.5428
4,4,0.8698,0.870,0.4163,8.8526


In [5]:
_ = step('TF-IDF + LinearSVC (calibrated)')

TF-IDF + LinearSVC (calibrated)  [Linear (TF-IDF)]  loaded saved results
macro-F1 0.8465 ± 0.0116   accuracy 0.8476   log loss 0.436   fit 0.8 s/fold   predict 0.13 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8344,0.838,0.4575,0.6915
1,1,0.8545,0.858,0.4247,0.6599
2,2,0.8443,0.846,0.4451,0.6480
3,3,0.8373,0.834,0.4137,0.6388
4,4,0.8620,0.862,0.4388,1.2076


In [6]:
_ = step('TF-IDF word+char + LinearSVC (calibrated)')

TF-IDF word+char + LinearSVC (calibrated)  [Linear (TF-IDF)]  loaded saved results
macro-F1 0.8475 ± 0.0162   accuracy 0.8484   log loss 0.430   fit 3.6 s/fold   predict 0.74 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8272,0.830,0.4518,3.6065
1,1,0.8514,0.854,0.4121,3.6090
2,2,0.8345,0.836,0.4462,3.5750
3,3,0.8655,0.862,0.4131,3.4862
4,4,0.8589,0.860,0.4285,3.7662


In [7]:
_ = step('TF-IDF + SGD (modified Huber)')

TF-IDF + SGD (modified Huber)  [Linear (TF-IDF)]  loaded saved results
macro-F1 0.8374 ± 0.0083   accuracy 0.8392   log loss 1.086   fit 0.3 s/fold   predict 0.10 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8435,0.846,1.2923,0.2789
1,1,0.8271,0.832,0.9896,0.2840
2,2,0.8297,0.832,1.1722,0.2729
3,3,0.8435,0.842,0.8156,0.2796
4,4,0.8433,0.844,1.1600,0.2783


In [8]:
_ = step('TF-IDF + Ridge')

TF-IDF + Ridge  [Linear (TF-IDF)]  loaded saved results
macro-F1 0.8500 ± 0.0100   accuracy 0.8508   log loss nan   fit 0.3 s/fold   predict 0.09 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8394,0.842,NaN,0.3185
1,1,0.8450,0.848,NaN,0.3147
2,2,0.8556,0.858,NaN,0.3438
3,3,0.8457,0.842,NaN,0.3285
4,4,0.8645,0.864,NaN,0.3015


## Probabilistic

In [9]:
_ = step('TF-IDF + Complement Naive Bayes')

TF-IDF + Complement Naive Bayes  [Probabilistic]  loaded saved results
macro-F1 0.8346 ± 0.0110   accuracy 0.8328   log loss 0.557   fit 0.3 s/fold   predict 0.10 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8218,0.822,0.5918,0.2587
1,1,0.8321,0.830,0.5454,0.2557
2,2,0.8327,0.832,0.5622,0.2485
3,3,0.8522,0.846,0.5214,0.3204
4,4,0.8340,0.834,0.5621,0.3313


In [10]:
_ = step('NB-SVM (NB-weighted LogReg, 1-3 grams)')

NB-SVM (NB-weighted LogReg, 1-3 grams)  [Probabilistic]  loaded saved results
macro-F1 0.8403 ± 0.0139   accuracy 0.8416   log loss 0.542   fit 5.4 s/fold   predict 0.15 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8168,0.820,0.6086,4.8646
1,1,0.8387,0.842,0.4601,5.5180
2,2,0.8454,0.848,0.6003,5.3781
3,3,0.8490,0.846,0.4538,5.6810
4,4,0.8514,0.852,0.5854,5.7772


## Tree ensemble

In [11]:
_ = step('LSA-300 + LightGBM')

LSA-300 + LightGBM  [Tree ensemble]  loaded saved results
macro-F1 0.8176 ± 0.0208   accuracy 0.8184   log loss 0.587   fit 12.9 s/fold   predict 0.20 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8066,0.810,0.6468,13.2004
1,1,0.8324,0.834,0.5437,12.7425
2,2,0.7877,0.790,0.6462,12.7731
3,3,0.8397,0.836,0.5232,13.2988
4,4,0.8216,0.822,0.5729,12.7028


## Linear (TF-IDF)

In [12]:
_ = step('LSA-300 + LogReg')

LSA-300 + LogReg  [Linear (TF-IDF)]  loaded saved results
macro-F1 0.8247 ± 0.0137   accuracy 0.8260   log loss 0.458   fit 4.0 s/fold   predict 0.21 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8068,0.810,0.4983,4.1753
1,1,0.8431,0.848,0.4356,4.0116
2,2,0.8292,0.830,0.4691,4.0667
3,3,0.8273,0.826,0.4104,4.0755
4,4,0.8169,0.816,0.4766,3.7022


## Ensemble

In [13]:
_ = step('Soft vote: word+char LogReg + cal. SVC + NB-SVM')

Soft vote: word+char LogReg + cal. SVC + NB-SVM  [Ensemble]  loaded saved results
macro-F1 0.8530 ± 0.0130   accuracy 0.8544   log loss 0.407   fit 17.7 s/fold   predict 1.63 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8301,0.834,0.4395,17.0052
1,1,0.8598,0.864,0.3834,15.6555
2,2,0.8543,0.856,0.4197,18.6623
3,3,0.8609,0.858,0.3842,18.4330
4,4,0.8597,0.860,0.4093,18.7346


## Word embeddings

In [14]:
_ = step('spaCy en_core_web_md vectors + LogReg')

spaCy en_core_web_md vectors + LogReg  [Word embeddings]  loaded saved results
macro-F1 0.7325 ± 0.0268   accuracy 0.7316   log loss 0.772   fit 6.5 s/fold   predict 3.71 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7342,0.732,0.8291,17.6887
1,1,0.7161,0.718,0.7349,3.6782
2,2,0.7086,0.712,0.7850,3.9455
3,3,0.7771,0.772,0.6584,3.6061
4,4,0.7266,0.724,0.8531,3.6170


In [15]:
_ = step('Gensim GloVe-300 mean + LogReg')

Gensim GloVe-300 mean + LogReg  [Word embeddings]  loaded saved results
macro-F1 0.7738 ± 0.0053   accuracy 0.7728   log loss 0.629   fit 40.8 s/fold   predict 0.20 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7785,0.776,0.6506,190.7205
1,1,0.7745,0.776,0.6509,3.1809
2,2,0.7657,0.766,0.6396,4.0339
3,3,0.7783,0.774,0.5903,2.9182
4,4,0.7719,0.772,0.6124,2.9728


In [16]:
_ = step('Gensim Doc2Vec (PV-DBOW) + LogReg')

Gensim Doc2Vec (PV-DBOW) + LogReg  [Word embeddings]  loaded saved results
macro-F1 0.7793 ± 0.0210   accuracy 0.7800   log loss 0.689   fit 52.1 s/fold   predict 2.59 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7557,0.756,0.7632,51.7123
1,1,0.7847,0.788,0.6900,53.0454
2,2,0.7605,0.764,0.6702,51.6647
3,3,0.8062,0.802,0.6391,52.0617
4,4,0.7894,0.790,0.6821,52.1713


In [17]:
_ = step('TF-IDF + GloVe-300 mean + LogReg')

TF-IDF + GloVe-300 mean + LogReg  [Word embeddings]  loaded saved results
macro-F1 0.7435 ± 0.0167   accuracy 0.7436   log loss 1.087   fit 87.4 s/fold   predict 0.26 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7334,0.734,1.2134,92.9079
1,1,0.7365,0.740,1.1503,85.0464
2,2,0.7263,0.728,1.1247,82.5119
3,3,0.7669,0.762,0.9245,91.3711
4,4,0.7544,0.754,1.0235,85.1584


## fastText

In [18]:
_ = step('fastText (scratch, bigrams)')

fastText (scratch, bigrams)  [fastText]  loaded saved results
macro-F1 0.7713 ± 0.0115   accuracy 0.7716   log loss 0.655   fit 1.7 s/fold   predict 0.04 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7581,0.758,0.7234,1.7141
1,1,0.7766,0.778,0.6313,1.7532
2,2,0.7607,0.766,0.6257,1.7175
3,3,0.7850,0.782,0.5776,1.7432
4,4,0.7763,0.774,0.7171,1.8196


In [19]:
_ = step('fastText (GloVe-300 init, bigrams)')

fastText (GloVe-300 init, bigrams)  [fastText]  loaded saved results
macro-F1 0.8171 ± 0.0138   accuracy 0.8180   log loss 0.709   fit 121.3 s/fold   predict 0.07 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7994,0.800,0.7958,121.5566
1,1,0.8264,0.830,0.6598,121.7206
2,2,0.8087,0.812,0.7273,121.3151
3,3,0.8343,0.832,0.6173,121.4287
4,4,0.8168,0.816,0.7436,120.6967


## Neural (from scratch)

In [20]:
_ = step('ANN / MLP on TF-IDF')

ANN / MLP on TF-IDF  [Neural (from scratch)]  loaded saved results
macro-F1 0.8469 ± 0.0141   accuracy 0.8484   log loss 0.438   fit 3.7 s/fold   predict 0.21 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8255,0.830,0.4685,7.6967
1,1,0.8489,0.852,0.4346,2.2372
2,2,0.8416,0.842,0.4603,2.2889
3,3,0.8613,0.860,0.3933,3.0867
4,4,0.8569,0.858,0.4350,2.9895


In [21]:
_ = step('CNN (Kim 2014) + GloVe')

CNN (Kim 2014) + GloVe  [Neural (from scratch)]  loaded saved results
macro-F1 0.8294 ± 0.0075   accuracy 0.8300   log loss 0.475   fit 2.6 s/fold   predict 0.06 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8266,0.826,0.5149,2.7464
1,1,0.8419,0.846,0.4234,2.4362
2,2,0.8220,0.824,0.4851,2.0427
3,3,0.8276,0.824,0.4559,2.9818
4,4,0.8287,0.830,0.4934,2.7332


In [22]:
_ = step('BiLSTM + attention + GloVe')

BiLSTM + attention + GloVe  [Neural (from scratch)]  loaded saved results
macro-F1 0.8100 ± 0.0187   accuracy 0.8108   log loss 0.530   fit 9.7 s/fold   predict 0.12 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7825,0.784,0.5349,8.0203
1,1,0.8161,0.822,0.4943,8.0015
2,2,0.8000,0.804,0.6073,11.7345
3,3,0.8229,0.816,0.5103,10.4087
4,4,0.8286,0.828,0.5007,10.4762


In [23]:
_ = step('Transformer encoder (2-layer, from scratch) + GloVe')

Transformer encoder (2-layer, from scratch) + GloVe  [Neural (from scratch)]  loaded saved results
macro-F1 0.7864 ± 0.0194   accuracy 0.7880   log loss 0.593   fit 4.0 s/fold   predict 0.10 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7882,0.786,0.6343,4.3298
1,1,0.7883,0.788,0.5452,3.4987
2,2,0.7632,0.770,0.6110,3.4019
3,3,0.8158,0.814,0.5166,4.2012
4,4,0.7764,0.782,0.6601,4.5072


## Transformer (frozen)

In [24]:
_ = step('MiniLM-L6 sentence embeddings + LogReg')

MiniLM-L6 sentence embeddings + LogReg  [Transformer (frozen)]  loaded saved results
macro-F1 0.7748 ± 0.0171   accuracy 0.7776   log loss 0.578   fit 6.0 s/fold   predict 0.79 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.7589,0.762,0.5876,12.4925
1,1,0.7787,0.784,0.5596,4.9345
2,2,0.7671,0.770,0.5859,4.5662
3,3,0.8026,0.804,0.5473,4.0615
4,4,0.7667,0.768,0.6120,4.1066


In [25]:
_ = step('MPNet-base sentence embeddings + LogReg')

MPNet-base sentence embeddings + LogReg  [Transformer (frozen)]  loaded saved results
macro-F1 0.8147 ± 0.0149   accuracy 0.8168   log loss 0.477   fit 6.2 s/fold   predict 2.68 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8013,0.804,0.4881,11.4953
1,1,0.7977,0.804,0.4641,4.6303
2,2,0.8228,0.824,0.4744,4.8689
3,3,0.8330,0.832,0.4641,4.3667
4,4,0.8189,0.820,0.4952,5.5450


In [26]:
_ = step('BGE-base sentence embeddings + LogReg')

BGE-base sentence embeddings + LogReg  [Transformer (frozen)]  loaded saved results
macro-F1 0.8190 ± 0.0114   accuracy 0.8200   log loss 0.478   fit 6.1 s/fold   predict 2.80 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8031,0.802,0.4949,10.7644
1,1,0.8309,0.834,0.4439,5.6119
2,2,0.8148,0.818,0.4845,4.4857
3,3,0.8293,0.828,0.4617,3.6498
4,4,0.8172,0.818,0.5048,6.0406


## Transformer (fine-tuned)

In [27]:
_ = step('DistilBERT-base fine-tuned')

DistilBERT-base fine-tuned  [Transformer (fine-tuned)]  loaded saved results
macro-F1 0.8629 ± 0.0085   accuracy 0.8660   log loss 0.419   fit 19.5 s/fold   predict 0.97 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8572,0.862,0.4467,20.2426
1,1,0.8581,0.864,0.4010,19.3135
2,2,0.8709,0.874,0.4204,19.4967
3,3,0.8733,0.872,0.3477,19.3177
4,4,0.8550,0.858,0.4796,19.3248


In [28]:
_ = step('BERT-base fine-tuned')

BERT-base fine-tuned  [Transformer (fine-tuned)]  loaded saved results
macro-F1 0.8669 ± 0.0106   accuracy 0.8700   log loss 0.413   fit 36.7 s/fold   predict 1.77 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8566,0.860,0.4684,37.5313
1,1,0.8691,0.874,0.3852,36.6254
2,2,0.8556,0.860,0.4267,36.4403
3,3,0.8801,0.880,0.3587,36.4271
4,4,0.8732,0.876,0.4265,36.5603


In [29]:
_ = step('FinBERT (ProsusAI) fine-tuned')

FinBERT (ProsusAI) fine-tuned: no saved results (set RUN_MISSING=1 to train it)


In [30]:
_ = step('RoBERTa-base fine-tuned')

RoBERTa-base fine-tuned  [Transformer (fine-tuned)]  loaded saved results
macro-F1 0.8755 ± 0.0185   accuracy 0.8788   log loss 0.392   fit 37.7 s/fold   predict 1.72 ms/doc


,fold,f1_macro,accuracy,log_loss,fit_s
0,0,0.8473,0.854,0.4357,38.4759
1,1,0.8716,0.876,0.4080,37.6100
2,2,0.8965,0.900,0.3441,37.4118
3,3,0.8862,0.886,0.3854,37.3711
4,4,0.8759,0.878,0.3883,37.4367


## Ranking

Models with all 5 folds, sorted by macro-F1. `p_vs_best` is the Nadeau-Bengio corrected resampled
t-test against the top model; p ≥ 0.05 means the gap is within fold-to-fold noise.

In [31]:
ranking = B.report(len(X), 5)
cols = ["model", "family", "f1_macro", "f1_std", "accuracy", "log_loss", "p_vs_best", "fit_s", "ms_per_doc", "size_kb"]
display(ranking[cols].round(4))

,model,family,f1_macro,f1_std,accuracy,log_loss,p_vs_best,fit_s,ms_per_doc,size_kb
0,RoBERTa-base fine-tuned,Transformer (fine-tuned),0.8755,0.0185,0.8788,0.3923,NaN,37.6611,1.7164,4.869090e+05
1,BERT-base fine-tuned,Transformer (fine-tuned),0.8669,0.0106,0.8700,0.4131,0.5386,36.7169,1.7702,4.276770e+05
2,DistilBERT-base fine-tuned,Transformer (fine-tuned),0.8629,0.0085,0.8660,0.4191,0.2413,19.5391,0.9667,2.615490e+05
3,Soft vote: word+char LogReg + cal. SVC + NB-SVM,Ensemble,0.8530,0.0130,0.8544,0.4072,0.0493,17.6981,1.6302,1.090241e+04
4,TF-IDF + Ridge,Linear (TF-IDF),0.8500,0.0100,0.8508,NaN,0.0717,0.3214,0.0946,8.348799e+02
5,TF-IDF word+char + LogReg,Linear (TF-IDF),0.8477,0.0155,0.8492,0.4218,0.1328,8.0965,0.7060,1.995909e+03
6,TF-IDF word+char + LinearSVC (calibrated),Linear (TF-IDF),0.8475,0.0162,0.8484,0.4303,0.0939,3.6086,0.7362,7.380809e+03
7,ANN / MLP on TF-IDF,Neural (from scratch),0.8469,0.0141,0.8484,0.4383,0.0453,3.6598,0.2117,4.189802e+04
8,TF-IDF + LinearSVC (calibrated),Linear (TF-IDF),0.8465,0.0116,0.8476,0.4359,0.0941,0.7692,0.1265,3.031015e+03
9,TF-IDF + LogReg (current production),Linear (TF-IDF),0.8446,0.0089,0.8460,0.4519,0.0402,5.6712,0.0979,8.401260e+02
